In [ ]:
!pip install ISLP

In [ ]:

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.utils import resample
from statsmodels.formula.api import ols
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
from sklearn.model_selection import KFold
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression
from sklearn import datasets
import matplotlib.pyplot as plt


In [ ]:
# Load the Boston dataset from ISLP
from ISLP import load_data
boston = load_data('Boston')

# Define features (X) and target (y)
X = boston.drop(columns='medv')
y = boston['medv']


In [ ]:
# Train-Test Split
np.random.seed(88888)
train = np.random.choice(len(boston), 300, replace=False)

# Train a Linear Regression model using statsmodels
X_train = sm.add_constant(X.iloc[train])
X_test = sm.add_constant(X.drop(train))

y_train = y.iloc[train]
y_test = y.drop(train)

lm_fit1 = sm.OLS(y_train, X_train).fit()

# Predict and calculate MSE on test set
lm1_pred = lm_fit1.predict(X_test)
mse = mean_squared_error(y_test, lm1_pred)
print(f"Test Set MSE: {mse}")


In [ ]:
# LOOCV
X_with_constant = sm.add_constant(X)
lm_fit_all = sm.OLS(y, X_with_constant).fit()

loo = KFold(n_splits=len(X))  # LOOCV = n splits
cv_mse = cross_val_score(LinearRegression(), X, y, cv=loo, scoring='neg_mean_squared_error')

mean_cv_mse = -np.mean(cv_mse)
print(f"LOOCV MSE: {mean_cv_mse}")


In [ ]:
# 10-Fold Cross-Validation
kf_10 = KFold(n_splits=10, shuffle=True, random_state=2)
cv_mse_10 = cross_val_score(LinearRegression(), X, y, cv=kf_10, scoring='neg_mean_squared_error')

mean_cv_mse_10 = -np.mean(cv_mse_10)
print(f"10-Fold Cross-Validation MSE: {mean_cv_mse_10}")

# 5-Fold Cross-Validation
kf_5 = KFold(n_splits=5, shuffle=True, random_state=2)
cv_mse_5 = cross_val_score(LinearRegression(), X, y, cv=kf_5, scoring='neg_mean_squared_error')

mean_cv_mse_5 = -np.mean(cv_mse_5)
print(f"5-Fold Cross-Validation MSE: {mean_cv_mse_5}")


In [ ]:
# Bootstrap to estimate the mean of medv
def bootstrap_mean(data, n_iterations=1000):
    means = []
    for _ in range(n_iterations):
        sample = resample(data)
        means.append(np.mean(sample))
    return means

bootstrap_means = bootstrap_mean(boston['medv'], n_iterations=1000)
print(f"Bootstrap Estimate for Mean of 'medv': {np.mean(bootstrap_means)}")


In [ ]:
# Fit the Linear Regression model using statsmodels
model = sm.OLS(y, X)
results = model.fit()

# Print the regression summary to get the standard errors from the model
print("OLS Regression Output:")
print(results.summary())

# Function for bootstrapping
def bootstrap_coefficients(data, target, n_bootstrap=1000):
    coefs = []
    n = len(data)

    for _ in range(n_bootstrap):
        # Resample the data with replacement
        X_resampled, y_resampled = resample(data, target)

        # Fit the model to the resampled data
        model_boot = sm.OLS(y_resampled, X_resampled)
        results_boot = model_boot.fit()

        # Store the coefficients
        coefs.append(results_boot.params.values)

    return np.array(coefs)

# Perform bootstrapping (1000 resamples)
n_bootstrap = 1000
bootstrap_coefs = bootstrap_coefficients(X, y, n_bootstrap=n_bootstrap)

# Calculate the mean and standard deviation of the bootstrapped coefficients
bootstrap_means = np.mean(bootstrap_coefs, axis=0)
bootstrap_stds = np.std(bootstrap_coefs, axis=0)

# Print the standard errors from OLS and Bootstrap side by side
print("\nCoefficient Comparison (OLS vs Bootstrap):")
for i, coef_name in enumerate(X.columns):
    print(f"{coef_name}:")
    print(f"    OLS Std Error: {results.bse[i]:.4f}")
    print(f"    Bootstrap Std Dev: {bootstrap_stds[i]:.4f}")